# 🎯 Target Locker — SAM2.1 Tiny (Google Colab)

**Single-Object Tracking (SOT): select one target and keep tracking only that target.**

This notebook is prepared for the GitHub project:
**https://github.com/CptImtiaz/Target_Locker**

### Input options
- **Upload video** from your computer
- **Open webcam in Colab** and record a short clip directly from your browser

> Colab runs on a remote machine, so `cv2.VideoCapture(0)` cannot directly access your laptop webcam.  
> The webcam mode below opens your browser camera, records a short clip, transfers it to Colab, and then runs the same SAM2 tracking pipeline.

### Run order
Run all cells from top to bottom. Use a **GPU runtime**:
`Runtime → Change runtime type → T4 GPU`.


## 1. Setup


In [ ]:
import os, sys, subprocess, tempfile, shutil, time, base64, json, uuid, urllib.request
from pathlib import Path

import torch
assert torch.cuda.is_available(), "Select Runtime → Change runtime type → GPU first."
print("GPU:", torch.cuda.get_device_name(0))

# Lightweight dependencies only. Keep Colab's own torch/torchvision build.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "hydra-core==1.3.2", "iopath==0.1.10", "tqdm", "pillow"],
    check=True
)

import cv2
import numpy as np
import torchvision

PROJECT_ROOT = Path("/content/Target_Locker")
SAM2_ROOT = PROJECT_ROOT / "SAM2_streaming-main"

# Clone the current GitHub repository instead of checking out an old hard-coded commit.
if not SAM2_ROOT.exists():
    if PROJECT_ROOT.exists():
        shutil.rmtree(PROJECT_ROOT)
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/CptImtiaz/Target_Locker.git",
         str(PROJECT_ROOT)],
        check=True
    )

assert SAM2_ROOT.exists(), f"Missing {SAM2_ROOT}. Check the GitHub repository structure."
os.chdir(SAM2_ROOT)

if str(SAM2_ROOT) not in sys.path:
    sys.path.insert(0, str(SAM2_ROOT))

from sam2.build_sam import build_sam2_camera_predictor

print("Repository:", PROJECT_ROOT)
print("SAM2 source:", SAM2_ROOT)
print("Setup complete.")


## 2. Choose input: Upload Video or Webcam

Change `INPUT_MODE` to:

- `"upload"` — upload an existing video
- `"webcam"` — open your browser webcam and record a short clip

For webcam mode, change `WEBCAM_SECONDS` if you want a longer recording.


In [ ]:
#@title Input settings
INPUT_MODE = "webcam"  #@param ["upload", "webcam"]
WEBCAM_SECONDS = 8     #@param {type:"integer"}
MAX_INPUT_SIDE = 640   #@param {type:"integer"}

assert INPUT_MODE in {"upload", "webcam"}
assert 2 <= WEBCAM_SECONDS <= 60, "Use 2–60 seconds for webcam recording."
assert 320 <= MAX_INPUT_SIDE <= 1280

from google.colab import files, output

target = None
output_path = None
video_path = None
source_fps = None
source_frames = None

def fit_frame(frame):
    h, w = frame.shape[:2]
    scale = min(1.0, MAX_INPUT_SIDE / max(h, w))
    nw = max(2, int(w * scale) // 2 * 2)
    nh = max(2, int(h * scale) // 2 * 2)
    if (nw, nh) == (w, h):
        return frame
    return cv2.resize(frame, (nw, nh), interpolation=cv2.INTER_AREA)

def load_video_info(path):
    cap = cv2.VideoCapture(str(path))
    ok, frame = cap.read()
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    assert ok, "Cannot decode video. Try MP4/H.264 or record again."
    frame = fit_frame(frame)
    if not np.isfinite(fps) or fps <= 0:
        fps = 25.0
    return frame, float(fps), frames

if INPUT_MODE == "upload":
    uploaded = files.upload()
    assert len(uploaded) == 1, "Select exactly one video."
    name, data = next(iter(uploaded.items()))
    suffix = Path(name).suffix or ".mp4"
    video_path = Path("/content") / ("input-" + uuid.uuid4().hex + suffix)
    video_path.write_bytes(data)
    del uploaded, data
    print("Uploaded:", name)

else:
    print(f"Webcam will open below and record for {WEBCAM_SECONDS} seconds.")


### 2A. Webcam recorder
Run this cell **only when `INPUT_MODE = "webcam"`**. Allow camera permission in your browser.


In [ ]:
if INPUT_MODE == "webcam":
    webcam_js = r"""
    async function recordWebcam(seconds) {
      const wrap = document.createElement('div');
      wrap.style.fontFamily = 'system-ui, sans-serif';
      wrap.style.padding = '12px';

      const title = document.createElement('div');
      title.innerHTML = '<b>🎥 Target Locker Webcam</b><br>Allow camera access. Recording starts automatically.';
      title.style.marginBottom = '10px';

      const video = document.createElement('video');
      video.autoplay = true;
      video.muted = true;
      video.playsInline = true;
      video.style.maxWidth = '720px';
      video.style.width = '100%';
      video.style.borderRadius = '12px';
      video.style.background = '#111';

      const status = document.createElement('div');
      status.style.marginTop = '10px';

      wrap.appendChild(title);
      wrap.appendChild(video);
      wrap.appendChild(status);
      document.body.appendChild(wrap);

      const stream = await navigator.mediaDevices.getUserMedia({
        video: {width: {ideal: 1280}, height: {ideal: 720}},
        audio: false
      });

      video.srcObject = stream;
      await video.play();

      const mimeCandidates = [
        'video/webm;codecs=vp9',
        'video/webm;codecs=vp8',
        'video/webm'
      ];
      let mimeType = '';
      for (const m of mimeCandidates) {
        if (MediaRecorder.isTypeSupported(m)) { mimeType = m; break; }
      }

      const recorder = new MediaRecorder(stream, mimeType ? {mimeType} : undefined);
      const chunks = [];

      recorder.ondataavailable = e => {
        if (e.data && e.data.size > 0) chunks.push(e.data);
      };

      const stopped = new Promise(resolve => recorder.onstop = resolve);

      recorder.start(250);

      for (let i = seconds; i > 0; i--) {
        status.textContent = `🔴 Recording... ${i}s`;
        google.colab.output.setIframeHeight(document.body.scrollHeight, true);
        await new Promise(r => setTimeout(r, 1000));
      }

      recorder.stop();
      await stopped;
      stream.getTracks().forEach(t => t.stop());
      status.textContent = '✅ Recording complete. Sending video to Colab...';

      const blob = new Blob(chunks, {type: recorder.mimeType || 'video/webm'});
      const dataUrl = await new Promise(resolve => {
        const reader = new FileReader();
        reader.onloadend = () => resolve(reader.result);
        reader.readAsDataURL(blob);
      });

      wrap.remove();
      return {dataUrl, mimeType: blob.type};
    }
    """

    result = output.eval_js(webcam_js + f"\nrecordWebcam({int(WEBCAM_SECONDS)})")
    data_url = result["dataUrl"]
    assert "," in data_url

    webm_path = Path("/content") / ("webcam-" + uuid.uuid4().hex + ".webm")
    webm_path.write_bytes(base64.b64decode(data_url.split(",", 1)[1]))

    # Convert browser WebM recording to H.264 MP4 for robust OpenCV decoding.
    mp4_path = Path("/content") / ("webcam-" + uuid.uuid4().hex + ".mp4")
    convert = subprocess.run(
        ["ffmpeg", "-nostdin", "-y",
         "-i", str(webm_path),
         "-an",
         "-c:v", "libx264", "-preset", "fast", "-crf", "22",
         "-pix_fmt", "yuv420p",
         "-movflags", "+faststart",
         str(mp4_path)],
        capture_output=True, text=True
    )
    if convert.returncode != 0:
        print(convert.stderr[-1500:])
        raise RuntimeError("Webcam recording conversion failed.")

    video_path = mp4_path
    print("Webcam clip ready:", video_path)
else:
    print("Upload mode selected — skip webcam recording.")


## 3. Read the input and prepare the first frame


In [ ]:
assert video_path is not None and Path(video_path).is_file(), \
    "No input video found. Run the input cell (and webcam recorder if using webcam mode)."

first_frame, source_fps, source_frames = load_video_info(video_path)
print(f"Input ready: {source_frames} frames @ {source_fps:.2f} FPS")
print("Next: click the target.")


## 4. Click the target, then press **Confirm target**

Click once **inside the object you want to lock onto**.  
A yellow circle will mark your selected point.


In [ ]:
from google.colab import output

target = None

ok, jpg = cv2.imencode(".jpg", first_frame)
assert ok
image_url = "data:image/jpeg;base64," + base64.b64encode(jpg).decode("ascii")

javascript = r"""
(async () => {
  const root = document.createElement('div');
  root.style.fontFamily = 'system-ui, sans-serif';
  root.style.padding = '12px';

  const tip = document.createElement('p');
  tip.innerHTML = '<b>🎯 Click inside the target</b>, then press Confirm target.';

  const canvas = document.createElement('canvas');
  canvas.style.maxWidth = '100%';
  canvas.style.cursor = 'crosshair';
  canvas.style.borderRadius = '10px';

  const button = document.createElement('button');
  button.textContent = 'Confirm target';
  button.disabled = true;
  button.style.marginTop = '10px';
  button.style.padding = '8px 14px';

  root.append(tip, canvas, document.createElement('br'), button);
  document.body.appendChild(root);

  const image = new Image();
  image.src = IMAGE_URL;
  await image.decode();

  canvas.width = image.naturalWidth;
  canvas.height = image.naturalHeight;

  const ctx = canvas.getContext('2d');
  ctx.drawImage(image, 0, 0);

  let selected = null;

  canvas.onclick = event => {
    const r = canvas.getBoundingClientRect();
    const x = Math.max(0, Math.min(canvas.width - 1,
      Math.floor((event.clientX - r.left) * canvas.width / r.width)));
    const y = Math.max(0, Math.min(canvas.height - 1,
      Math.floor((event.clientY - r.top) * canvas.height / r.height)));

    selected = [x, y];

    ctx.drawImage(image, 0, 0);
    ctx.strokeStyle = '#FFD400';
    ctx.fillStyle = 'rgba(255,212,0,.18)';
    ctx.lineWidth = 4;
    ctx.beginPath();
    ctx.arc(x, y, 9, 0, 2 * Math.PI);
    ctx.fill();
    ctx.stroke();

    button.disabled = false;
  };

  google.colab.output.setIframeHeight(document.body.scrollHeight, true);

  const point = await new Promise(resolve => {
    button.onclick = () => {
      if (selected) resolve(selected);
    };
  });

  button.disabled = true;
  canvas.onclick = null;
  tip.innerHTML = '✅ Target saved. Continue to the model cell.';
  return point;
})()
""".replace("IMAGE_URL", json.dumps(image_url))

point = output.eval_js(javascript)
x, y = map(int, point)

assert 0 <= x < first_frame.shape[1] and 0 <= y < first_frame.shape[0]
target = {"video": str(video_path), "point": [x, y]}

print("Target saved:", target["point"])


## 5. Load SAM2.1 Tiny on GPU


In [ ]:
checkpoint = Path("/content/sam2.1_hiera_tiny.pt")

if not checkpoint.exists():
    print("Downloading SAM2.1 Tiny checkpoint...")
    temp = checkpoint.with_suffix(".download")
    urllib.request.urlretrieve(
        "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt",
        temp
    )
    temp.replace(checkpoint)

if "predictor" not in globals():
    predictor = build_sam2_camera_predictor(
        "sam2.1/sam2.1_hiera_t.yaml",
        str(checkpoint),
        device="cuda"
    )

predictor.fill_hole_area = 0
amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Model ready:", torch.cuda.get_device_name(0))
print("Autocast:", amp_dtype)


## 6. Track the selected target

`MAX_FRAMES` controls how much of the clip is processed.

For the complete clip, set a larger value than the number of input frames.


In [ ]:
from tqdm.auto import tqdm

MAX_FRAMES = 300  #@param {type:"integer"}
assert isinstance(MAX_FRAMES, int) and MAX_FRAMES > 0
assert target and target["video"] == str(video_path), "Select and confirm the target first."

output_path = None
job_dir = Path(tempfile.mkdtemp(prefix="target-locker-", dir="/content"))
raw_path = job_dir / "tracked_raw.mp4"
final_path = job_dir / "tracked.mp4"

cap = cv2.VideoCapture(str(video_path))
writer = None
count = 0

try:
    ok, frame = cap.read()
    assert ok, "Cannot read video."
    frame = fit_frame(frame)

    h, w = frame.shape[:2]
    writer = cv2.VideoWriter(
        str(raw_path),
        cv2.VideoWriter_fourcc(*"mp4v"),
        source_fps,
        (w, h)
    )
    assert writer.isOpened(), "Cannot create output video."

    total = min(MAX_FRAMES, source_frames) if source_frames > 0 else MAX_FRAMES

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode(), torch.autocast("cuda", dtype=amp_dtype):
        predictor.frame_idx = 0

        predictor.load_first_frame(
            cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        )

        _, _, logits = predictor.add_new_prompt(
            frame_idx=0,
            obj_id=1,
            points=np.array([target["point"]], dtype=np.float32),
            labels=np.array([1], dtype=np.int32)
        )

        for index in tqdm(range(total), desc="Target locking"):
            if index:
                ok, frame = cap.read()
                if not ok:
                    break

                frame = fit_frame(frame)
                _, logits = predictor.track(
                    cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                )

            mask = (logits[0, 0] > 0).detach().cpu().numpy()

            # Yellow mask overlay
            overlay = frame.copy()
            overlay[mask] = (0, 220, 255)
            result = cv2.addWeighted(frame, 0.68, overlay, 0.32, 0)

            # Bounding box around current mask
            ys, xs = np.where(mask)
            if xs.size:
                x1, x2 = int(xs.min()), int(xs.max())
                y1, y2 = int(ys.min()), int(ys.max())
                cv2.rectangle(result, (x1, y1), (x2, y2), (0, 255, 255), 2)
                cv2.putText(
                    result, "LOCKED TARGET",
                    (x1, max(24, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0, 255, 255), 2, cv2.LINE_AA
                )

            writer.write(result)
            count += 1

    torch.cuda.synchronize()
    seconds = time.perf_counter() - start

finally:
    cap.release()
    if writer is not None:
        writer.release()

    # Reset streaming state for another run.
    try:
        predictor.condition_state = {}
        predictor.frame_idx = 0
    except Exception:
        pass

    torch.cuda.empty_cache()

assert count > 0, "No frames were processed."
print(f"Processed {count} frames in {seconds:.1f}s — {count/seconds:.2f} processing FPS.")

# Browser-compatible H.264 output
encode = subprocess.run(
    ["ffmpeg", "-nostdin", "-y",
     "-i", str(raw_path),
     "-an",
     "-c:v", "libx264", "-preset", "fast", "-crf", "23",
     "-pix_fmt", "yuv420p",
     "-movflags", "+faststart",
     str(final_path)],
    capture_output=True, text=True
)

if encode.returncode:
    print("H.264 encoding failed; using raw MP4.")
    print(encode.stderr[-1000:])
    output_path = raw_path
else:
    output_path = final_path

print("Output:", output_path)


## 7. Preview and download


In [ ]:
from IPython.display import Video, display
from google.colab import files

assert output_path and Path(output_path).is_file(), "Finish tracking first."

size_mb = Path(output_path).stat().st_size / (1024 * 1024)
print(f"Output size: {size_mb:.1f} MB")

if size_mb < 50:
    display(Video(str(output_path), embed=True))
else:
    print("Output is large; downloading instead of embedding.")

files.download(str(output_path))


---

## Notes

- **Webcam mode in Colab is browser-recorded, then processed by the GPU runtime.**
- It is not continuous low-latency live streaming because the Colab GPU runs remotely.
- For true real-time webcam tracking with direct camera access, use the local **SAM2-Mac** version in this repository.
- The tracker does not need a separate detector after you initialize the target by clicking it.
